In [1]:
from transformers import pipeline
import pyttsx3
import time
import os

# Load emotion classifier
classifier = pipeline(
    task="text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None
)


e:\7. Projects From Sem 3\TTS\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 202.02it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:

def init_engine():
    engine = pyttsx3.init('sapi5')
    voices = engine.getProperty("voices")
    soft_voice_id = voices[1].id if len(voices) > 1 else voices[0].id
    default_voice_id = voices[0].id
    return engine, soft_voice_id, default_voice_id

def classify_emotion(text):
    scores = sorted(classifier(text)[0], key=lambda x: x["score"], reverse=True)
    top = scores[0]
    
    print(f"\n🎭 TESTING: '{text}'")
    print("All emotions:")
    for r in scores:
        print(f"  {r['label']:10s} -> {r['score']:.3f}")
    print(f"📊 TOP: {top['label']} ({top['score']:.2f})")
    print("-" * 60)
    
    return top["label"], top["score"]

def speak_joy(engine, soft_voice_id, text, filename):
    base_rate = engine.getProperty("rate")
    joy_rate = max(160, min(190, base_rate + 10))
    engine.setProperty("voice", soft_voice_id)
    engine.setProperty("rate", joy_rate)
    engine.setProperty("volume", 0.95)
    print("🌟 JOY SETTINGS: rate={} wpm, vol=0.95".format(joy_rate))
    save_emotional_speech(engine, text, filename)

def speak_anger(engine, default_voice_id, text, filename):
    base_rate = engine.getProperty("rate")
    anger_rate = max(210, min(240, base_rate + 50))
    engine.setProperty("voice", default_voice_id)
    engine.setProperty("rate", anger_rate)
    engine.setProperty("volume", 1.0)
    print("🔥 ANGER SETTINGS: rate={} wpm, vol=1.0".format(anger_rate))
    save_emotional_speech(engine, text, filename)

def speak_sadness(engine, soft_voice_id, text, filename):
    base_rate = engine.getProperty("rate")
    sad_rate = max(120, min(150, base_rate - 50))
    engine.setProperty("voice", soft_voice_id)
    engine.setProperty("rate", sad_rate)
    engine.setProperty("volume", 0.6)
    print("😢 SADNESS SETTINGS: rate={} wpm, vol=0.6".format(sad_rate))
    save_emotional_speech(engine, text, filename)

def speak_fear(engine, soft_voice_id, text, filename):
    """FEAR: NOW INTENSE - Very fast + panicked (240-280 wpm + MAX volume)"""
    base_rate = engine.getProperty("rate")
    fear_rate = max(240, min(280, base_rate + 80))  # **+80 = SUPER FAST**
    engine.setProperty("voice", soft_voice_id)
    engine.setProperty("rate", fear_rate)
    engine.setProperty("volume", 1.0)  # **MAX VOLUME**
    print("😱 FEAR INTENSE: rate={} wpm, vol=1.0 (PANICKED)".format(fear_rate))
    save_emotional_speech(engine, text, filename)

def speak_surprise(engine, soft_voice_id, text, filename):
    """SURPRISE: NOW INTENSE - Ultra-fast + shocked (230-270 wpm + piercing volume)"""
    base_rate = engine.getProperty("rate")
    surprise_rate = max(230, min(270, base_rate + 70))  # **+70 = ULTRA FAST**
    engine.setProperty("voice", soft_voice_id)
    engine.setProperty("rate", surprise_rate)
    engine.setProperty("volume", 1.0)  # **MAX VOLUME**
    print("😲 SURPRISE INTENSE: rate={} wpm, vol=1.0 (SHOCKED)".format(surprise_rate))
    save_emotional_speech(engine, text, filename)

def speak_disgust(engine, default_voice_id, text, filename):
    base_rate = engine.getProperty("rate")
    disgust_rate = max(130, min(160, base_rate - 25))
    engine.setProperty("voice", default_voice_id)
    engine.setProperty("rate", disgust_rate)
    engine.setProperty("volume", 0.8)
    print("🤢 DISGUST SETTINGS: rate={} wpm, vol=0.8".format(disgust_rate))
    save_emotional_speech(engine, text, filename)

def speak_neutral(engine, default_voice_id, text, filename):
    engine.setProperty("voice", default_voice_id)
    engine.setProperty("rate", 170)
    engine.setProperty("volume", 0.85)
    print("➖ NEUTRAL SETTINGS: rate=170 wpm, vol=0.85")
    save_emotional_speech(engine, text, filename)

def save_emotional_speech(engine, text, filename):
    """Save emotional speech to WAV file ONLY (NO PLAYBACK)"""
    if os.path.exists(filename):
        os.remove(filename)  # Clear old file
    print(f"💾 Saving '{text[:30]}...' → {filename}")
    engine.save_to_file(text, filename)
    engine.runAndWait()
    print(f"✅ SAVED: {filename}")

def speak_with_emotion(text, filename="output.wav"):
    emotion, confidence = classify_emotion(text)
    
    engine, soft_voice_id, default_voice_id = init_engine()
    
    emotion_handlers = {
        "joy": lambda: speak_joy(engine, soft_voice_id, text, filename),
        "anger": lambda: speak_anger(engine, default_voice_id, text, filename),
        "sadness": lambda: speak_sadness(engine, soft_voice_id, text, filename),
        "fear": lambda: speak_fear(engine, soft_voice_id, text, filename),
        "surprise": lambda: speak_surprise(engine, soft_voice_id, text, filename),
        "disgust": lambda: speak_disgust(engine, default_voice_id, text, filename),
        "neutral": lambda: speak_neutral(engine, default_voice_id, text, filename)
    }
    
    handler = emotion_handlers.get(emotion, emotion_handlers["neutral"])
    handler()
    print(f"\n🎵 FILE READY: {filename}")

# 🔥 INTENSE FEAR/SURPRISE TESTING
if __name__ == "__main__":
    print("🚀 EMPATHY ENGINE - INTENSE FEAR/SURPRISE TEST")
    print("=" * 70)
    
    intense_tests = [
        ("Oh no! There's a spider! Help me quick NOW!", "fear_intense.wav"),
        ("Wow! I can't believe you did that! AMAZING!", "surprise_intense.wav"),
        ("You can go to HELL! What are you doing?!", "anger_test.wav"),  # Reference
    ]
    
    for text, filename in intense_tests:
        speak_with_emotion(text, filename)
        print("\n" + "="*70 + "\n")
    
    print("🎉 INTENSE FILES SAVED!")
    print("📁 Check: fear_intense.wav, surprise_intense.wav")
    print("\n🎧 Play these - FEAR/SURPRISE should now be SUPER INTENSE!")


🚀 EMPATHY ENGINE - INTENSE FEAR/SURPRISE TEST

🎭 TESTING: 'Oh no! There's a spider! Help me quick NOW!'
All emotions:
  surprise   -> 0.707
  disgust    -> 0.085
  sadness    -> 0.068
  anger      -> 0.065
  neutral    -> 0.034
  fear       -> 0.025
  joy        -> 0.017
📊 TOP: surprise (0.71)
------------------------------------------------------------
😲 SURPRISE INTENSE: rate=270 wpm, vol=1.0 (SHOCKED)
💾 Saving 'Oh no! There's a spider! Help ...' → fear_intense.wav
✅ SAVED: fear_intense.wav

🎵 FILE READY: fear_intense.wav



🎭 TESTING: 'Wow! I can't believe you did that! AMAZING!'
All emotions:
  surprise   -> 0.943
  joy        -> 0.039
  neutral    -> 0.008
  anger      -> 0.004
  fear       -> 0.003
  sadness    -> 0.002
  disgust    -> 0.002
📊 TOP: surprise (0.94)
------------------------------------------------------------
😲 SURPRISE INTENSE: rate=270 wpm, vol=1.0 (SHOCKED)
💾 Saving 'Wow! I can't believe you did t...' → surprise_intense.wav
✅ SAVED: surprise_intense.wav

🎵 FILE 